# Data Preprocessing Sanity check
## Choose dataset, preprocessing method, and review results

This notebook provides a unified interface to:
1. Load data
2. Choose a preprocessing method (basic, **Llama**, or **Gemini API**)
3. **Test with a small sample first** (NUM_SAMPLES = 5)
4. Review and compare results
5. Run on full dataset when satisfied

## Configuration

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
import pprint
import google.generativeai as genai
from dotenv import load_dotenv

# Add project root to path
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)  
sys.path.append(project_root)
sys.path.append(os.path.join(project_root, "src"))  # Add src to path


try:
    from mosaic.preprocessing.translation_utils import (
        list_preprocessed_datasets,
        TruncationDiagnostic,
        compare_word_counts,
        compare_source_and_translated,
        show_preprocessing_stats,
        resolve_data_path
    )
except ImportError as e:
    print(f"Error loading modules: {e}")


print("Preprocessing module loaded successfully")

Preprocessing module loaded successfully


## Check inventory of available (preprocessed) datasets

In [2]:
# Auto-detect all available preprocessed data
inventory = list_preprocessed_datasets()

if "error" in inventory:
    print(f"Error: {inventory['error']}")
else:
    print(f"Found {len(inventory)} datasets with preprocessed files:\n")
    for name, data in inventory.items():
        print(f"{name} ({data['total_reports']} reports)")
        for f in data['files']:
            print(f"   - {f['filename']} ({f['size_mb']} MB) [{f['method'].upper()}]")
        print()

Found 5 datasets with preprocessed files:

5MeO_naturalistic (9197 reports)
   - 5MeO_naturalistic_preprocessed.csv (0.7 MB) [BASIC]

dreamachine_DL (98 reports)
   - dreamachine_DL_preprocessed.csv (0.03 MB) [BASIC]

dreamachine_HS (334 reports)
   - dreamachine_HS_preprocessed.csv (0.09 MB) [BASIC]

ganzfeld_GREEN (34 reports)
   - ganzfeld_GREEN_cleaned_llama.csv (0.05 MB) [LLAMA]

ganzfeld_RED (34 reports)
   - ganzfeld_RED_cleaned_llama.csv (0.06 MB) [LLAMA]



## Configutation: select Dataset and Preprocessing Method



In [3]:
# ========================================================
# CONFIGURATION
# ========================================================

# Paste the filename you want to check (e.g. 'MPE_cleaned_API.csv')
TARGET_FILE = 'NDE_preprocessed.csv'

# Column names (usually these defaults work)
SOURCE_COL = 'reflection_answer'
TARGET_COL = 'phen_report_english' #'cleaned_reflection'

# ========================================================

# FIX: Add 'DATA/' to the path string
full_path = resolve_data_path(f"DATA/preprocessed/{TARGET_FILE}")

if full_path.exists():
    print(f"Selected file: {TARGET_FILE}")
    print(f"   Path: {full_path}")
else:
    print(f"File not found: {TARGET_FILE}")
    print(f"   Checked path: {full_path}")
    print("Check the filename from the list above.")

File not found: NDE_preprocessed.csv
   Checked path: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv
Check the filename from the list above.


In [4]:
#Checks row counts and looks for an associated error log.
show_preprocessing_stats(str(full_path))


PREPROCESSING STATISTICS

[ERROR] Data file not found: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv
Tried:
  - /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv
  - /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv
  - /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv
Project root: /Users/rb666/Projects/MOSAIC


In [5]:
#check truncation: Calculates retention rates and checks for cut-off text.
diagnostic = TruncationDiagnostic(
    str(full_path), 
    source_col=SOURCE_COL, 
    target_col=TARGET_COL
)

diagnostic.run_full_diagnostic()

[ERROR] Could not load CSV: [Errno 2] No such file or directory: '/Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv'
Tried: /Users/rb666/Projects/MOSAIC/DATA/preprocessed/NDE_preprocessed.csv
Project root: /Users/rb666/Projects/MOSAIC
[ERROR] Could not load data. Aborting diagnostic.


In [6]:
pd.set_option('display.max_colwidth', None)  # No truncation
pd.set_option('display.width', None)

#visual comaparison of translation vs original
compare_source_and_translated(
    diagnostic.df, 
    source_col=SOURCE_COL, 
    target_col=TARGET_COL, 
    num_samples=3,
    anonymise=False  # Set to True to mask names/PII in output
)

AttributeError: 'TruncationDiagnostic' object has no attribute 'df'

In [ ]:
diagnostic.df[[SOURCE_COL, TARGET_COL]].head(10)

reflection_answer  \
0  Mon expérience se déroule en plusieurs expériences toutes reliées entre elles pour atteindre leur dénouement final. Tout d’abord il faut expliquer le contexte dans lequel ce que j’ai vécu s’est déroulé. Je suis un ex-toxicomane, j’ai consommé durant 15 ans à peu près toutes les drogues (douces, chimiques, etc.)  Et les derniers 5 ans en consommant des drogues «dures» (héroïne (peu) et par la suite cocaïne en injection (beaucoup). Cela m’entraîna dans une déchéance de plus en plus profonde et une «SOUFFRANCE» immense et insoutenable. Je suis issu d’un milieu favorisé de classe moyenne «aisé», élevé depuis l’âge de trois mois par ma grand-mère, femme spirituelle, de grande expérience de la vie, (une sainte, quoi !). Puis après le remariage de ma mère ce fut la séparation de ma grand-mère et une dépression s’ensuivit qui m’amena à l’idée de suicide vers l’âge de 12 ans, et qui m’amena à consommer de la drogue pour la première fois. Dépression et drogue ne faisant pas bon ménage, mon état se détériora jusqu’à la mort de mes grands-parents et après je connus la déchéance TOTALE : itinérance, tentatives de suicide (dizaines de tentatives en près de deux ans), état dépressif, consommation de drogues «dures», thérapies et rechutes, et autres tentatives de suicide. C’est lors d’une de ces tentatives que je connus ce que j’appelle une « EXPÉRIENCE DE RENCONTRE DIVINE»  au cours de laquelle DIEU se manifesta à moi et que ma vie «bascula» complètement et instantanément. J’avais fait ma deuxième thérapie et j’avais «rechuté» dans la consommation de drogue (cocaïne en injection) ; je considérais mon état comme «SANS ISSUE» et je n’espérais plus trouver de solution pour me sortir de cette déchéance et de cette «SOUFFRANCE» devenue INSUPPORTABLE. Plusieurs fois je suis tombé à genoux et les bras tendus vers le ciel  j’ai demandé à Dieu de venir me «chercher» afin de me délivrer de ma souffrance. J’avais fait plusieurs tentatives de suicide, certaines étaient plus des appels à ’aide, mais cette fois, je ne croyais plus à aucune aide autre que celle de Dieu. Je me rendis chez un ami (souffrant de maniaco-dépression avancée et profonde) qui devait prendre des médicaments très forts (antidépresseurs qui à cette époque n’avaient pas la particularité d’être anti-suicide comme aujourd’hui) et j’avalai ce qui restai de la bouteille de pilules et je bus la moitié d’une bouteille de cognac. \n\nOn me transporta à l’hôpital après m’avoir trouvé dans un état de semi-conscience. Je perdis conscience durant le transport. Tout-à-coup je m’éveillai sur une civière, aux soins intensifs. Cela semblait être le soir car on n’entendait aucun bruit (ou peu) et pas de vas et viens. L’éclairage où j’étais était baissé et une infirmière se tenait dos à moi et préparait des piluliers sur un comptoir, éclairé par une veilleuse. Je voyais que j’étais relié par des fils à des machines (cardiogramme et autres), j’avais un tube dans la gorge qui me faisait souffrir et qui m’empêchait de fermer la bouche. J’avais soif terriblement et je me suis assis dans mon lit et je demandai à l’infirmière à boire. Elle ne m’entendait pas, ne me répondait pas ! Je parlai plus fort et même chose, puis elle se retourna et parla à une autre infirmière derrière moi, quelque chose n’ayant pas rapport avec moi. Je croyais qu’elle m’ignorait. Exaspéré, je décidai de me lever pour avoir à boire et descendis de mon lit. J’étais debout, à côté de la civière et en regardant la civière, quelqu’un était couché dedans, MOI! Je savais que j’étais sorti de mon corps mais ne voulais pas le croire et je réitérai ma demande à boire en me plaçant tout près de l’infirmière et je lui ai CRIÉ « À BOIRE ». Toujours pas de réponse ! Puis, comme un voile épais et noir, comme une lumière qui s’éteint, ce «noir» s’abattit dans la pièce. À ce moment l’infirmière se retourna et regarda le cardiogramme et dit à quelqu’un «on est en train de le perdre ! » Ce fut les dernières paroles que j’eus l